In [ ]:
pip install ultralytics

from ultralytics import YOLO
model = YOLO("yolov8m.pt")

%%writefile data.yaml

train: /content/drive/MyDrive/Datasets/vision/VisDrone/VisDrone2019-DET-train/images
val: /content/drive/MyDrive/Datasets/vision/VisDrone/VisDrone2019-DET-val/images

nc: 10

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor


In [ ]:
model.train(
    data="data.yaml",
    epochs=100,
    patience=20,
    batch=8,
    imgsz=640,
    lr0=0.001,
    weight_decay=0.0005,
    degrees=5,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    cache=False,
    workers=2,
    amp=True,
    project="/content/drive/MyDrive/ObjectDetection",
    name="visdrone_yolov8m"
)

In [ ]:
result = model.predict(
    source = "/content/drive/MyDrive/Datasets/vision/VisDrone/VisDrone2019-DET-val/images",
    conf = 0.7,
    save = True
)

In [ ]:
model = YOLO(
    "/content/drive/MyDrive/ObjectDetection/visdrone_detector-100/weights/best.pt"
)

In [ ]:
results = model.predict(
    source="video.mp4",
    conf=0.8,
    save=True
)

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

video_path = "/content/drive/MyDrive/Datasets/test video.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

output_path = "/content/output_detection.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break


    results = model(frame)

    annotated_frame = results[0].plot()

    out.write(annotated_frame)

    cv2_imshow(annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()

print("Video saved at:")
print(output_path)